In [ ]:
# Enable automatic timing for all cells in this notebook

import time
from IPython import get_ipython

_cell_start_time = None


def _pre_run_cell(info):
    global _cell_start_time
    _cell_start_time = time.time()


def _post_run_cell(result):
    global _cell_start_time
    if _cell_start_time is None:
        return
    elapsed = time.time() - _cell_start_time
    print(f"[Cell executed in {elapsed:.3f} seconds]")


_ip = get_ipython()
if _ip is not None:
    # Register hooks so every subsequent cell is timed automatically
    _ip.events.register("pre_run_cell", _pre_run_cell)
    _ip.events.register("post_run_cell", _post_run_cell)
else:
    print("[Timing hooks not installed: no active IPython shell detected]")


# Redoing some alanysis to align with SQL analysis

## Data Import 
Importing small sample of data to use as a prototype. Here, we will be using python with the pandas library to attain a performance baseline.

In [3]:
import pandas as pd

df = pd.read_csv("datasets/cyclistic_tripdata_2021.csv", nrows=10000)
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,31FAE55254BEE127,classic_bike,2021-02-12 09:40:15 UTC,2021-02-12 13:28:31 UTC,Fairbanks Ct & Grand Ave,TA1305000003,NaN,NaN,41.891847,-87.620580,NaN,NaN,casual
1,ED658F5C9D645C49,classic_bike,2021-02-14 18:41:23 UTC,2021-02-15 14:08:36 UTC,Clark St & Wrightwood Ave,TA1305000014,NaN,NaN,41.929546,-87.643118,NaN,NaN,member
2,AB91DE991455DBCB,classic_bike,2021-02-14 17:45:13 UTC,2021-02-15 18:45:08 UTC,Clark St & Lake St,KA1503000012,NaN,NaN,41.886021,-87.630876,NaN,NaN,member
3,6E37DC248B3CFE21,classic_bike,2021-02-20 14:34:21 UTC,2021-02-21 15:34:15 UTC,Columbus Dr & Randolph St,13263,NaN,NaN,41.884728,-87.619521,NaN,NaN,member
4,985662404D86B374,classic_bike,2021-02-27 13:22:20 UTC,2021-02-28 14:22:14 UTC,Clark St & Drummond Pl,TA1307000142,NaN,NaN,41.931248,-87.644336,NaN,NaN,casual


In [4]:
df["started_at"] = pd.to_datetime(df["started_at"], utc=True)
df["ended_at"] = pd.to_datetime(df["ended_at"], utc=True)
df["ride_length"] = df["ended_at"] - df["started_at"]
df["ride_length_minutes"] = df["ride_length"].dt.total_seconds() / 60
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_length,ride_length_minutes
0,31FAE55254BEE127,classic_bike,2021-02-12 09:40:15+00:00,2021-02-12 13:28:31+00:00,Fairbanks Ct & Grand Ave,TA1305000003,NaN,NaN,41.891847,-87.620580,NaN,NaN,casual,0 days 03:48:16,228.266667
1,ED658F5C9D645C49,classic_bike,2021-02-14 18:41:23+00:00,2021-02-15 14:08:36+00:00,Clark St & Wrightwood Ave,TA1305000014,NaN,NaN,41.929546,-87.643118,NaN,NaN,member,0 days 19:27:13,1167.216667
2,AB91DE991455DBCB,classic_bike,2021-02-14 17:45:13+00:00,2021-02-15 18:45:08+00:00,Clark St & Lake St,KA1503000012,NaN,NaN,41.886021,-87.630876,NaN,NaN,member,1 days 00:59:55,1499.916667
3,6E37DC248B3CFE21,classic_bike,2021-02-20 14:34:21+00:00,2021-02-21 15:34:15+00:00,Columbus Dr & Randolph St,13263,NaN,NaN,41.884728,-87.619521,NaN,NaN,member,1 days 00:59:54,1499.900000
4,985662404D86B374,classic_bike,2021-02-27 13:22:20+00:00,2021-02-28 14:22:14+00:00,Clark St & Drummond Pl,TA1307000142,NaN,NaN,41.931248,-87.644336,NaN,NaN,casual,1 days 00:59:54,1499.900000


## Making Time and Day cols
Allows for us to perform day and time analysis

In [10]:
df["hour"] = df["started_at"].dt.hour
df["day_of_week"] = df["started_at"].dt.day_name()

df[["started_at", "hour", "day_of_week"]].head()

,started_at,hour,day_of_week
0,2021-02-12 09:40:15+00:00,9,Friday
1,2021-02-14 18:41:23+00:00,18,Sunday
2,2021-02-14 17:45:13+00:00,17,Sunday
3,2021-02-20 14:34:21+00:00,14,Saturday
4,2021-02-27 13:22:20+00:00,13,Saturday


## Bike type usage
Here, we wanted to find the relationship between bike type and user type. We found the average ride duration for each bike and user combination. This analysis helps identify user preferences for specific bike types and highlights the differences between members and casual riders that we have previously theorised about. 
The shear number of member usage is higher, but when we look at the percentage usage, the figures start to make sense. Members seem to prefer the "classic bike" over the electric more so than the casual users do, this could be down to members being into fitness or simply the bike availability in their residential areas.
The only question we are left with right now is why do only casual users use "docked bikes"?

In [6]:
bike_user_ct = pd.crosstab(df["rideable_type"], df["member_casual"])
print("Bike type x user type (counts):")
print(bike_user_ct)


print("\nBike type x user type (row %):")
bike_pct = bike_user_ct.div(bike_user_ct.sum(axis=1), axis=0).round(3)
print(bike_pct)

print("\nAverage ride length (minutes) by bike type and user type:")

avg_duration_bike_user = (
    df.groupby(["rideable_type", "member_casual"])["ride_length_minutes"]
    .mean()
    .round(1)
    .unstack()
)
print(avg_duration_bike_user)

print("\nInterpretation by bike type:")
for bike in bike_user_ct.index:
    casual_share = bike_pct.loc[bike, "casual"] * 100
    member_share = bike_pct.loc[bike, "member"] * 100
    print(f"- {bike}: {casual_share:.1f}% casual, {member_share:.1f}% member")

Bike type x user type (counts):
member_casual  casual  member
rideable_type                
classic_bike     1037    5846
docked_bike       251       0
electric_bike     704    2162

Bike type x user type (row %):
member_casual  casual  member
rideable_type                
classic_bike    0.151   0.849
docked_bike     1.000   0.000
electric_bike   0.246   0.754

Average ride length (minutes) by bike type and user type:
member_casual  casual  member
rideable_type                
classic_bike     32.6    15.1
docked_bike     182.9     NaN
electric_bike    15.5    12.6

Interpretation by bike type:
- classic_bike: 15.1% casual, 84.9% member
- docked_bike: 100.0% casual, 0.0% member
- electric_bike: 24.6% casual, 75.4% member


## Station-based insights
We wanted to find out which stations had the highest numbers of users arriving and departing from them, we can see that "Dearborn St and Erie St" appears one of the focal points, seeing as it is the most popular destination and second most popular starting point. Similar observations can be made for each other station.
We also computed the average ride duration by start station an user type. We filtered this to include stations with 30+ trips to ensure reliability.
Using this information, we can detarmine what areas are residential and which are tourist hotspots among many other things.

In [7]:
for user_type in ["member", "casual"]:
    print(f"\nTop 10 start stations for {user_type}s:")
    top_start = (
        df[df["member_casual"] == user_type]["start_station_name"]
        .value_counts()
        .head(10)
    )
    print(top_start)

    print(f"\nTop 10 end stations for {user_type}s:")
    top_end = (
        df[df["member_casual"] == user_type]["end_station_name"]
        .value_counts()
        .head(10)
    )
    print(top_end)

min_trips_per_station = 30

avg_duration_station = (
    df.groupby(["start_station_name", "member_casual"])["ride_length_minutes"]
    .agg(["count", "mean"])
    .reset_index()
)



avg_duration_station = avg_duration_station[avg_duration_station["count"] >= min_trips_per_station]

print("\nAverage ride length (minutes) by start station and user type (>= 30 trips):")
print(avg_duration_station.sort_values("mean", ascending=False).head(20))


Top 10 start stations for members:
start_station_name
Dearborn St & Erie St        82
Clark St & Elm St            78
Wells St & Elm St            78
Wells St & Huron St          74
Desplaines St & Kinzie St    68
Kingsbury St & Kinzie St     67
Clinton St & Madison St      66
Columbus Dr & Randolph St    65
Daley Center Plaza           64
St. Clair St & Erie St       64
Name: count, dtype: int64

Top 10 end stations for members:
end_station_name
Dearborn St & Erie St        102
Clark St & Elm St             93
St. Clair St & Erie St        76
Wabash Ave & Roosevelt Rd     73
Wells St & Elm St             65
Kingsbury St & Kinzie St      63
Clinton St & Madison St       62
Broadway & Barry Ave          59
McClurg Ct & Erie St          58
Wells St & Concord Ln         56
Name: count, dtype: int64

Top 10 start stations for casuals:
start_station_name
Lake Shore Dr & Monroe St    29
Ellis Ave & 60th St          25
Millennium Park              25
Streeter Dr & Grand Ave      19
Daley Cen

## Time of day x day of week bike usage
We could use seaborn to display this information on a heatmap as it is very suited for that, but seeing as we are attempting to set a base line for time, we thought it best to stick with text outputs in case graphics take a significantly long amount of time to display.
Now bike usage by day of week and time is legible. The huge numbers of users visible during the work rush hours on weekdays confirms our earlier hypotheses. The low numbers of rides over the weekend do so as well, likely being mostly tourist use.

In [12]:
#this is basically a text heat map, I wanted to make something easily duplicated in other languages without importing libraries
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
cat_type = pd.CategoricalDtype(categories=day_order, ordered=True)
df["day_of_week"] = df["day_of_week"].astype(cat_type)

for user_type in ["member", "casual"]:
    subset = df[df["member_casual"] == user_type]
    pivot = subset.pivot_table(
        index="day_of_week",
        columns="hour",
        values="ride_id",
        aggfunc="count",
        fill_value=0,
    )

    print(f"\nTrips table (day x hour) for {user_type}:")
    print(pivot)

    print(f"\nTop 5 day-hour cells for {user_type}:")
    top_cells = (
        pivot.stack()
        .sort_values(ascending=False)
        .head(5)
        .reset_index(name="trips")
    )
    for _, row in top_cells.iterrows():
        print(f"- {row['day_of_week']} @ {row['hour']}: {row['trips']} trips")


Trips table (day x hour) for member:
hour         0   1   2   3   4   5   6   7   8   9   ...   14   15   16   17  \
day_of_week                                          ...                       
Monday        3   1   4   1   1  15  48  65  53  38  ...   85   66   97   94   
Tuesday       3   1   0   1   1  18  42  80  56  49  ...   64   93  112  130   
Wednesday     2   0   2   2   3  18  52  87  81  39  ...   79  101  126  116   
Thursday      4   2   0   4   3  15  44  82  84  56  ...   67   71  108  141   
Friday        6   3   1   2   3  24  48  73  92  64  ...   74  108  101  139   
Saturday      9   5   6   2   1   3  16  27  44  60  ...  143  125   99  106   
Sunday       14   6   3   3   1   3   6  16  25  30  ...   95   91   64   62   

hour          18  19  20  21  22  23  
day_of_week                           
Monday        95  57  31  12  14  10  
Tuesday       94  59  34  16  18   6  
Wednesday    107  70  48  17  13   6  
Thursday     109  51  36  22   9  12  
Friday 

## Analysing Daily Trends
we can see saturday is extra busy for casual users which supports the theory they are tourists trying to view the city

In [15]:
weekday_counts = (
    df.groupby(["day_of_week", "member_casual"])
    .size()
    .unstack(fill_value=0)
    .reindex(day_order)
)

member_counts = weekday_counts["member"]
casual_counts = weekday_counts["casual"]


print("Trips by day of week and user type (counts):")
print(weekday_counts)

print("\nDay-of-week share of trips by user type (row %):")
weekday_pct = weekday_counts.div(weekday_counts.sum(axis=1), axis=0).round(3)
print(weekday_pct)

print("\nKey weekday patterns:")

for day in weekday_counts.index:
    casual = weekday_counts.loc[day, "casual"]
    member = weekday_counts.loc[day, "member"]
    print(f"- {day}: {casual} casual vs {member} member trips")

Trips by day of week and user type (counts):
member_casual  casual  member
day_of_week                  
Monday            202    1000
Tuesday           197    1125
Wednesday         217    1248
Thursday          241    1162
Friday            287    1293
Saturday          531    1315
Sunday            317     865

Day-of-week share of trips by user type (row %):
member_casual  casual  member
day_of_week                  
Monday          0.168   0.832
Tuesday         0.149   0.851
Wednesday       0.148   0.852
Thursday        0.172   0.828
Friday          0.182   0.818
Saturday        0.288   0.712
Sunday          0.268   0.732

Key weekday patterns:
- Monday: 202 casual vs 1000 member trips
- Tuesday: 197 casual vs 1125 member trips
- Wednesday: 217 casual vs 1248 member trips
- Thursday: 241 casual vs 1162 member trips
- Friday: 287 casual vs 1293 member trips
- Saturday: 531 casual vs 1315 member trips
- Sunday: 317 casual vs 865 member trips


## Busy Stations
We can see the busiest stations, ranked on the number of arrivals + departures

In [17]:
departures = df.groupby("start_station_name").size().rename("departures")
arrivals = df.groupby("end_station_name").size().rename("arrivals")


station_activity = (
    departures.to_frame()
    .join(arrivals.to_frame(), how="outer")
    .fillna(0)
)

station_activity["total_activity"] = (
    station_activity["departures"] + station_activity["arrivals"]
)



top_stations = station_activity.sort_values("total_activity", ascending=False).head(10)

print("Top stations by total activity (departures + arrivals):")
print(top_stations)

Top stations by total activity (departures + arrivals):
                           departures  arrivals  total_activity
start_station_name                                             
Clark St & Elm St                96.0     112.0           208.0
Dearborn St & Erie St            94.0     112.0           206.0
Wells St & Elm St                93.0      83.0           176.0
Wells St & Huron St              91.0      61.0           152.0
St. Clair St & Erie St           71.0      81.0           152.0
Clinton St & Madison St          78.0      71.0           149.0
Kingsbury St & Kinzie St         79.0      69.0           148.0
Desplaines St & Kinzie St        79.0      64.0           143.0
Broadway & Waveland Ave          73.0      69.0           142.0
Columbus Dr & Randolph St        80.0      56.0           136.0


## Ride lengths by bike type
We can see what the average ride length is for both casuals and members for each type of bike.

In [20]:

valid = df[df["ride_length_minutes"].notna()].copy()

avg_duration_bike_user = (valid.groupby(["rideable_type", "member_casual"])["ride_length_minutes"].mean().round(1).unstack())

print("Average ride length (minutes) by bike type and user type:")
print(avg_duration_bike_user)


Average ride length (minutes) by bike type and user type:
member_casual  casual  member
rideable_type                
classic_bike     32.6    15.1
docked_bike     182.9     NaN
electric_bike    15.5    12.6
